---
title: "Tutorial: From Sound to Syllables"
subtitle: Detecting and describing syllables in juvenile zebra finch song
authors:
  - name: Sheetal Jadhav
---

**Estimated time:** 60 to 90 minutes

## Tutorial objectives

A young zebra finch practises its song for weeks before the song becomes stable. To study that learning, we first have to turn a microphone recording into *units* we can count and compare. In this tutorial you will:

1. Visualise a recording as a waveform and a spectrogram, and see how the window length trades time against frequency resolution.
2. Detect sound events with an amplitude threshold.
3. Check the detector against syllables that an expert labelled by hand.
4. Describe each event with acoustic features and use them to pick out one syllable type.
5. Measure the rhythm of that syllable.

:::{note} About the data
We use one minute of song from a juvenile zebra finch recorded in the Mooney lab [@qi2026juvenile]. The audio was denoised before release. The hand labels mark the onsets and offsets of **one** recurring syllable type, not every sound the bird made. Keep this in mind: it shapes how we can evaluate our tools.
:::

## Setup

In [ ]:
# Imports
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio, display

In [ ]:
# @title Figure settings
plt.rcParams.update({
    "figure.figsize": (10, 3),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

In [ ]:
# @title Data retrieval
ZENODO = "https://zenodo.org/records/20411580/files/"
RECORDING = "recording_20220323_084711nrd"


def find_data_dir():
    # Use the book's data/ folder when it exists, otherwise ./data (e.g. on Colab)
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "myst.yml").exists():
            return folder / "data"
    return Path("data")


def fetch(name, data_dir):
    # Download a file from Zenodo once and reuse the local copy afterwards
    path = data_dir / name
    if not path.exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(ZENODO + name + "?download=1", path)
    return path


DATA_DIR = find_data_dir()
wav_path = fetch(RECORDING + ".wav", DATA_DIR)
label_path = fetch(RECORDING + ".txt", DATA_DIR)

In [ ]:
# @title Plotting helpers
def shade_labels(ax, labels, color="tab:red", alpha=0.25):
    # Shade hand-labelled syllables on an axis
    for onset, offset in labels:
        ax.axvspan(onset, offset, color=color, alpha=alpha, lw=0)


def plot_spectrogram(ax, f, t, Sxx, tlim=None, fmax=10_000, vmin=-110, vmax=-40):
    # Plot a power spectrogram in decibels, frequency in kHz
    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    keep = np.ones_like(t, dtype=bool) if tlim is None else (t >= tlim[0]) & (t <= tlim[1])
    ax.pcolormesh(t[keep], f / 1000, Sxx_db[:, keep], shading="auto",
                  vmin=vmin, vmax=vmax, cmap="magma")
    ax.set_ylim(0, fmax / 1000)
    ax.set_ylabel("Frequency (kHz)")
    ax.set_xlabel("Time (s)")

---
## Section 1: Load and listen

A `.wav` file stores the microphone's pressure signal as a long list of numbers, sampled tens of thousands of times per second. We load it, scale it to the range −1 to 1, and read the hand labels: one row per syllable, with onset and offset in seconds.

In [ ]:
fs, audio = wavfile.read(wav_path)
audio = audio.astype(float) / np.abs(audio).max()
labels = np.loadtxt(label_path)  # columns: onset (s), offset (s)

print(f"Sampling rate: {fs} Hz")
print(f"Duration: {len(audio) / fs:.1f} s")
print(f"Hand-labelled syllables: {len(labels)}")

Listen to the first four seconds. You will hear soft noisy sounds, upward sweeps and a short whistle-like note that repeats.

In [ ]:
display(Audio(audio[: 4 * fs], rate=fs))

In [ ]:
#| label: fig-waveform
#| caption: The first 3.5 s of the recording. Red bands mark the hand-labelled syllables.
time = np.arange(len(audio)) / fs
tlim = (0, 3.5)
keep = (time >= tlim[0]) & (time <= tlim[1])

fig, ax = plt.subplots()
ax.plot(time[keep], audio[keep], lw=0.5, color="k")
shade_labels(ax, labels[labels[:, 0] < tlim[1]])
ax.set(xlabel="Time (s)", ylabel="Amplitude", xlim=tlim)
plt.show()

:::{admonition} Think!
:class: tip
Look at [](#fig-waveform). Can you tell the labelled syllables apart from the other sounds using the waveform alone? What information is missing?
:::

---
## Section 2: Spectrograms and the time-frequency trade-off

A spectrogram cuts the signal into short, overlapping windows and computes the power at each frequency in each window. The window length $N$ (in samples) sets the resolution:

$$
\Delta f = \frac{f_s}{N}, \qquad \Delta t \approx \frac{N}{f_s}
$$

A long window resolves frequency finely but blurs timing; a short window does the opposite. You cannot have both.

:::{exercise} Compute a spectrogram
:label: ex-spectrogram
Complete `compute_spectrogram` so that it converts `window_ms` to a number of samples and calls `scipy.signal.spectrogram` with 75% overlap between windows. It should return the frequencies, times and power.
:::

In [ ]:
def compute_spectrogram(audio, fs, window_ms=8):
    # TODO: convert window_ms to samples (an integer)
    nperseg = ...
    # TODO: call signal.spectrogram with nperseg and 75% overlap
    f, t, Sxx = ...
    return f, t, Sxx

:::{solution} ex-spectrogram
:class: dropdown
The solution is in the hidden cell below.
:::

In [ ]:
def compute_spectrogram(audio, fs, window_ms=8):
    nperseg = int(window_ms / 1000 * fs)
    f, t, Sxx = signal.spectrogram(audio, fs, nperseg=nperseg, noverlap=nperseg * 3 // 4)
    return f, t, Sxx

In [ ]:
#| label: fig-windows
#| caption: The same 1.2 s of song with three window lengths. Short windows sharpen onsets; long windows sharpen pitch.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, window_ms in zip(axes, [2, 8, 32]):
    f, t, Sxx = compute_spectrogram(audio, fs, window_ms)
    plot_spectrogram(ax, f, t, Sxx, tlim=(0.4, 1.6))
    ax.set_title(f"{window_ms} ms window, Δf = {fs / int(window_ms / 1000 * fs):.0f} Hz")
plt.tight_layout()
plt.show()

For the rest of the tutorial we use an 8 ms window, a common compromise for zebra finch song.

In [ ]:
#| label: fig-overview
#| caption: First 3.5 s as a spectrogram. The labelled syllable is the short tone near 5 kHz.
f, t, Sxx = compute_spectrogram(audio, fs, window_ms=8)

fig, ax = plt.subplots(figsize=(12, 3.5))
plot_spectrogram(ax, f, t, Sxx, tlim=(0, 3.5))
shade_labels(ax, labels[labels[:, 0] < 3.5], color="cyan", alpha=0.2)
plt.show()

---
## Section 3: Detecting sound events

A simple detector has three steps:

1. **Envelope**: sum the power in a frequency band where song lives (here 3 to 8 kHz) and convert to decibels. This ignores most low-frequency cage noise.
2. **Threshold**: mark every time bin where the envelope is above a threshold. We set the threshold relative to the *noise floor*, the median envelope, because the absolute level depends on the microphone.
3. **Clean up**: join events separated by very short gaps and drop events that are too short to be syllables.

In [ ]:
def band_envelope_db(f, Sxx, f_low=3000, f_high=8000):
    # Total power between f_low and f_high in each time bin, in dB
    band = (f >= f_low) & (f <= f_high)
    return 10 * np.log10(Sxx[band].sum(axis=0) + 1e-12)


envelope = band_envelope_db(f, Sxx)
noise_floor = np.median(envelope)
print(f"Noise floor: {noise_floor:.1f} dB")

:::{exercise} Threshold the envelope
:label: ex-detect
Complete `detect_events`. Given time bins `t`, an envelope and a threshold, return arrays of onset and offset times. Merge events separated by less than `min_gap` seconds and remove events shorter than `min_dur` seconds.

*Hint:* `np.diff` of a boolean array (as integers, padded with zeros) is +1 at onsets and −1 just after offsets.
:::

In [ ]:
def detect_events(t, envelope, threshold, min_dur=0.02, min_gap=0.01):
    above = envelope > threshold
    # TODO: find the indices where `above` switches on and off
    # TODO: merge events closer than min_gap, then drop events shorter than min_dur
    ...
    return onsets, offsets

:::{solution} ex-detect
:class: dropdown
The solution is in the hidden cell below.
:::

In [ ]:
def detect_events(t, envelope, threshold, min_dur=0.02, min_gap=0.01):
    above = np.r_[0, (envelope > threshold).astype(int), 0]
    edges = np.diff(above)
    starts = t[np.where(edges == 1)[0]]
    stops = t[np.where(edges == -1)[0] - 1]

    onsets, offsets = [], []
    for start, stop in zip(starts, stops):
        if onsets and start - offsets[-1] < min_gap:
            offsets[-1] = stop          # close to the previous event: merge
        else:
            onsets.append(start)
            offsets.append(stop)
    onsets, offsets = np.array(onsets), np.array(offsets)

    long_enough = (offsets - onsets) >= min_dur
    return onsets[long_enough], offsets[long_enough]

In [ ]:
#| label: fig-detection
#| caption: Envelope (black) with the threshold (dashed). Grey bars show detected events; red bands show the hand labels.
threshold = noise_floor + 25
onsets, offsets = detect_events(t, envelope, threshold)
print(f"Detected {len(onsets)} events in {t[-1]:.0f} s")

tlim = (0, 3.5)
keep = (t >= tlim[0]) & (t <= tlim[1])
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[keep], envelope[keep], color="k", lw=0.8)
ax.axhline(threshold, ls="--", color="gray")
for on, off in zip(onsets, offsets):
    if on < tlim[1]:
        ax.plot([on, off], [-25, -25], color="gray", lw=6, solid_capstyle="butt")
shade_labels(ax, labels[labels[:, 0] < tlim[1]])
ax.set(xlim=tlim, ylim=(-125, -18), xlabel="Time (s)", ylabel="Band power (dB)")
plt.show()

---
## Section 4: Checking the detector against hand labels

How good is our detector? Since the labels only cover one syllable type, we cannot count false alarms: an unlabelled event might be a real sound the expert chose not to label. We *can* ask how well we recover the labelled syllables.

For each labelled syllable we find the detected event that overlaps it best, measured by **intersection over union (IoU)**: the overlapping time divided by the total time covered by both. An IoU above 0.5 counts as a hit.

:::{exercise} Match events to labels
:label: ex-iou
Complete `best_iou`, which returns, for every labelled syllable, the highest IoU with any detected event.
:::

In [ ]:
def best_iou(onsets, offsets, labels):
    scores = []
    for label_on, label_off in labels:
        # TODO: overlap = length of the intersection with each event (0 if none)
        # TODO: union = length of the span covered by the label and each event
        ...
    return np.array(scores)

:::{solution} ex-iou
:class: dropdown
The solution is in the hidden cell below.
:::

In [ ]:
def best_iou(onsets, offsets, labels):
    scores = []
    for label_on, label_off in labels:
        overlap = np.clip(np.minimum(offsets, label_off) - np.maximum(onsets, label_on), 0, None)
        union = np.maximum(offsets, label_off) - np.minimum(onsets, label_on)
        scores.append((overlap / union).max() if len(onsets) else 0.0)
    return np.array(scores)


iou = best_iou(onsets, offsets, labels)
print(f"Labels recovered (IoU > 0.5): {(iou > 0.5).sum()} of {len(labels)}")
print(f"Mean IoU: {iou.mean():.2f}")

Now sweep the threshold to see how the choice matters.

In [ ]:
#| label: fig-threshold
#| caption: Detector performance as the threshold rises above the noise floor. Too low and syllables merge with neighbouring sounds; too high and the faint target syllable is missed.
offsets_db = np.arange(5, 46, 2.5)
recall, mean_iou = [], []
for offset_db in offsets_db:
    on, off = detect_events(t, envelope, noise_floor + offset_db)
    scores = best_iou(on, off, labels)
    recall.append((scores > 0.5).mean())
    mean_iou.append(scores.mean())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(offsets_db, recall, "o-", label="Recall (IoU > 0.5)")
ax.plot(offsets_db, mean_iou, "s--", label="Mean IoU")
ax.set(xlabel="Threshold above noise floor (dB)", ylabel="Score", ylim=(0, 1.05))
ax.legend()
plt.show()

:::{admonition} Think!
:class: tip
1. Why does performance fall at *both* low and high thresholds? Look back at [](#fig-detection).
2. We tuned the threshold on the same labels we test on. Why is that a problem, and how would you fix it?
:::

---
## Section 5: Describing syllables with features

Detection gives us about a hundred events, but only some are the labelled syllable. To tell them apart we describe each event with numbers:

* **Duration** (s)
* **Peak frequency**: the median, across time bins, of the frequency with most power (Hz)
* **Wiener entropy**: how noisy versus tonal the sound is [@tchernichovski2000procedure]. For a power spectrum $P(f)$,

$$
H = \log_{10} \frac{\exp\left(\frac{1}{K}\sum_f \ln P(f)\right)}{\frac{1}{K}\sum_f P(f)}
$$

the log of the geometric mean over the arithmetic mean across $K$ frequency bins. White noise has $H \approx 0$; a pure tone is strongly negative.

:::{exercise} Wiener entropy
:label: ex-entropy
Complete `wiener_entropy` for a 1D power spectrum.
:::

In [ ]:
def wiener_entropy(power):
    # TODO: geometric mean / arithmetic mean, then log10
    ...

:::{solution} ex-entropy
:class: dropdown
The solution is in the hidden cell below.
:::

In [ ]:
def wiener_entropy(power):
    power = power + 1e-20
    geometric_mean = np.exp(np.mean(np.log(power)))
    return np.log10(geometric_mean / np.mean(power))


print(f"White noise: {wiener_entropy(np.ones(100)):.2f}")
print(f"A pure tone: {wiener_entropy(np.r_[1.0, np.zeros(99)]):.2f}")

In [ ]:
def syllable_features(f, t, Sxx, onsets, offsets, f_low=3000, f_high=8000):
    band = (f >= f_low) & (f <= f_high)
    rows = []
    for on, off in zip(onsets, offsets):
        frames = (t >= on) & (t <= off)
        power = Sxx[band][:, frames]
        peak_freq = np.median(f[band][power.argmax(axis=0)])
        rows.append([off - on, peak_freq, wiener_entropy(power.mean(axis=1))])
    return np.array(rows)


features = syllable_features(f, t, Sxx, onsets, offsets)
is_labelled = best_iou(labels[:, 0], labels[:, 1], np.c_[onsets, offsets]) > 0.5
print(f"{len(features)} events, {is_labelled.sum()} overlap a hand label")

In [ ]:
#| label: fig-features
#| caption: Every detected event in feature space. Labelled syllables (red) sit in a tonal band near 5 kHz.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(features[~is_labelled, 1] / 1000, features[~is_labelled, 2],
           s=20 + 300 * features[~is_labelled, 0], c="lightgray", edgecolor="gray", label="Other events")
ax.scatter(features[is_labelled, 1] / 1000, features[is_labelled, 2],
           s=20 + 300 * features[is_labelled, 0], c="tab:red", edgecolor="k", label="Hand-labelled")
ax.set(xlabel="Peak frequency (kHz)", ylabel="Wiener entropy (more tonal ↓)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
plt.show()

Marker size shows duration. A simple rule picks out the target: peak frequency between 4.5 and 5.6 kHz, and entropy below a cut-off. Here we can compute precision too, because we are asking a narrower question: *of the events the rule calls the target syllable, how many did the expert label?*

In [ ]:
in_band = (features[:, 1] > 4500) & (features[:, 1] < 5600)
print(f"{'Entropy cut-off':>16} {'Predicted':>10} {'Precision':>10} {'Recall':>8}")
for cutoff in [-1.6, -1.8, -2.0, -2.2]:
    predicted = in_band & (features[:, 2] < cutoff)
    hits = (predicted & is_labelled).sum()
    precision = hits / max(predicted.sum(), 1)
    recall_rule = hits / is_labelled.sum()
    print(f"{cutoff:>16} {predicted.sum():>10} {precision:>10.2f} {recall_rule:>8.2f}")

:::{admonition} Think!
:class: tip
A stricter cut-off raises precision but lowers recall. If you were studying how this syllable changes over development, which error would hurt you more, and why?
:::

---
## Section 6: Rhythm

Behaviour is not just *what* an animal does but *when*. The interval between successive onsets, the {abbr}`IOI (inter-onset interval)`, reveals the tempo of song. We compute it from the hand labels, keeping only intervals within a bout (shorter than 1.5 s).

In [ ]:
#| label: fig-ioi
#| caption: Intervals between successive onsets of the labelled syllable within bouts.
ioi = np.diff(labels[:, 0])
within_bout = ioi[ioi < 1.5]

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(within_bout * 1000, bins=12, color="tab:blue", edgecolor="w")
ax.set(xlabel="Inter-onset interval (ms)", ylabel="Count")
plt.show()
print(f"Median IOI: {np.median(within_bout) * 1000:.0f} ms (n = {len(within_bout)})")

The syllable recurs at a regular tempo inside each bout: an early sign of the stereotyped motif that will emerge as the bird learns.

---
## Summary

You went from a pressure wave to behavioural measurements:

1. A **spectrogram** with an 8 ms window balanced timing and pitch.
2. An **amplitude threshold** set relative to the noise floor detected sound events; its value mattered a lot.
3. **IoU** against hand labels showed how well we recovered the target syllable.
4. **Features** (peak frequency, Wiener entropy, duration) separated the target from other sounds, with a precision and recall trade-off.
5. **Inter-onset intervals** measured the song's tempo.

## Bonus

* The dataset has recordings from six days of this bird's development. Change `RECORDING` to `recording_20220328_095438nrd` (five days later) and rerun the notebook. Does the target syllable become more tonal or more regular?
* If you are running the notebook in Jupyter or Colab, use the slider below to explore the detection threshold interactively.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    @interact(offset_db=FloatSlider(value=25, min=5, max=45, step=2.5, description="dB above floor"))
    def explore_threshold(offset_db):
        on, off = detect_events(t, envelope, noise_floor + offset_db)
        scores = best_iou(on, off, labels)
        print(f"{len(on)} events | recall {(scores > 0.5).mean():.2f} | mean IoU {scores.mean():.2f}")
except ImportError:
    print("Install ipywidgets to use the interactive slider.")